install web app

In [33]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime

# ==========================================
# 1. ตั้งค่าเริ่มต้นให้ระบบจำข้อมูล (Session State)
# ==========================================
if 'bookings' not in st.session_state:
    st.session_state.bookings = []
    st.session_state.next_id = 1

# ==========================================
# 2. การตั้งค่าหน้าเว็บหลัก
# ==========================================
st.set_page_config(page_title="ระบบจองสนามกีฬา", page_icon="🏟️", layout="wide")
st.title("🏟️ ระบบจัดการจองสนามกีฬา")

# ==========================================
# 3. เมนูด้านข้าง (Sidebar Menu)
# ==========================================
menu = ["เพิ่มข้อมูลการจอง", "แสดงรายการทั้งหมด", "ค้นหาข้อมูล", "แก้ไขข้อมูล", "ลบข้อมูล"]
choice = st.sidebar.selectbox("📌 เลือกเมนูการทำงาน", menu)

# ==========================================
# 4. ฟังก์ชันการทำงานแต่ละเมนู
# ==========================================

# --- เมนู: แสดงรายการทั้งหมด ---
if choice == "แสดงรายการทั้งหมด":
    st.subheader("📋 รายการจองทั้งหมดในระบบ")
    if not st.session_state.bookings:
        st.info("📭 ยังไม่มีข้อมูลการจองในระบบ")
    else:
        df = pd.DataFrame(st.session_state.bookings)
        df.columns = ["รหัสการจอง", "ชื่อผู้จอง", "ประเภทสนาม", "วันที่จอง", "เวลาที่จอง"]
        st.dataframe(df, use_container_width=True, hide_index=True)

# --- เมนู: เพิ่มข้อมูลการจอง ---
elif choice == "เพิ่มข้อมูลการจอง":
    st.subheader("📝 เพิ่มข้อมูลการจองใหม่")

    with st.form("add_booking_form"):
        col1, col2 = st.columns(2)
        with col1:
            name = st.text_input("ชื่อผู้จอง *")
            field_type = st.selectbox("ประเภทสนาม *", ["ฟุตบอล", "ฟุตซอล", "แบดมินตัน", "บาสเกตบอล", "เทนนิส", "วอลเลย์บอล"])
        with col2:
            date = st.date_input("วันที่จอง")
            time = st.time_input("เวลาที่จอง")

        submitted = st.form_submit_button("✅ บันทึกการจอง")

        if submitted:
            if name.strip() == "":
                st.error("❌ กรุณากรอกชื่อผู้จอง")
            else:
                # แปลงวันที่และเวลาเป็นข้อความเพื่อนำไปเช็ค
                str_date = date.strftime("%Y-%m-%d")
                str_time = time.strftime("%H:%M")

                # --- [ส่วนที่เพิ่มใหม่] ตรวจสอบการจองซ้อน ---
                is_duplicate = False
                for b in st.session_state.bookings:
                    # เช็คว่า ประเภทสนาม วันที่ และเวลา ตรงกับที่มีคนจองไว้แล้วหรือไม่
                    if b["field_type"] == field_type and b["date"] == str_date and b["time"] == str_time:
                        is_duplicate = True
                        break # ถ้าเจอว่าซ้ำแล้ว ให้ออกจากลูปได้เลย

                if is_duplicate:
                    # ถ้าซ้ำ แจ้งเตือนและไม่บันทึก
                    st.error(f"❌ ขออภัย สนาม {field_type} วันที่ {str_date} เวลา {str_time} น. มีผู้จองไปแล้ว กรุณาเลือกเวลาหรือสนามอื่น")
                else:
                    # ถ้าไม่ซ้ำ ให้บันทึกข้อมูลตามปกติ
                    new_booking = {
                        "booking_id": st.session_state.next_id,
                        "name": name,
                        "field_type": field_type,
                        "date": str_date,
                        "time": str_time
                    }
                    st.session_state.bookings.append(new_booking)
                    st.success(f"🎉 บันทึกสำเร็จ! รหัสการจองของคุณคือ: **{st.session_state.next_id}**")
                    st.session_state.next_id += 1

# --- เมนู: ค้นหาข้อมูล ---
elif choice == "ค้นหาข้อมูล":
    st.subheader("🔍 ค้นหาข้อมูลการจอง")
    keyword = st.text_input("กรอก 'รหัสการจอง' หรือ 'ชื่อผู้จอง'")

    if st.button("ค้นหา"):
        if keyword:
            found = [b for b in st.session_state.bookings if str(keyword).lower() in str(b["booking_id"]).lower() or keyword.lower() in b["name"].lower()]
            if found:
                df_found = pd.DataFrame(found)
                df_found.columns = ["รหัสการจอง", "ชื่อผู้จอง", "ประเภทสนาม", "วันที่จอง", "เวลาที่จอง"]
                st.success(f"พบข้อมูล {len(found)} รายการ")
                st.dataframe(df_found, use_container_width=True, hide_index=True)
            else:
                st.warning(f"❌ ไม่พบข้อมูลที่ตรงกับ '{keyword}'")
        else:
            st.error("กรุณากรอกคำค้นหา")

# --- เมนู: แก้ไขข้อมูล ---
elif choice == "แก้ไขข้อมูล":
    st.subheader("✏️ แก้ไขเวลาหรือประเภทสนาม")
    booking_id = st.number_input("กรอก 'รหัสการจอง' ที่ต้องการแก้ไข", min_value=1, step=1)

    booking_to_edit = next((b for b in st.session_state.bookings if b["booking_id"] == booking_id), None)

    if booking_to_edit:
        st.info(f"ผู้จอง: {booking_to_edit['name']} | ข้อมูลเดิม: สนาม{booking_to_edit['field_type']} เวลา {booking_to_edit['time']}")

        with st.form("edit_booking_form"):
            new_field_type = st.selectbox("เลือกประเภทสนามใหม่", ["ฟุตบอล", "ฟุตซอล", "แบดมินตัน", "บาสเกตบอล", "เทนนิส", "วอลเลย์บอล"], index=["ฟุตบอล", "ฟุตซอล", "แบดมินตัน", "บาสเกตบอล", "เทนนิส", "วอลเลย์บอล"].index(booking_to_edit['field_type']))
            new_time = st.time_input("เลือกเวลาใหม่", value=datetime.strptime(booking_to_edit['time'], "%H:%M").time())

            update_submitted = st.form_submit_button("อัปเดตข้อมูล")
            if update_submitted:
                # หมายเหตุ: ในระบบจริงควรเช็คการจองซ้ำในขั้นตอนนี้ด้วย (แต่เบื้องต้นทำให้อัปเดตได้ไปก่อน)
                booking_to_edit["field_type"] = new_field_type
                booking_to_edit["time"] = new_time.strftime("%H:%M")
                st.success(f"✅ อัปเดตข้อมูลรหัส {booking_id} เรียบร้อยแล้ว")
    else:
        if st.session_state.bookings:
            st.warning("ไม่พบรหัสการจองนี้ในระบบ")

# --- เมนู: ลบข้อมูล ---
elif choice == "ลบข้อมูล":
    st.subheader("🗑️ ลบข้อมูลการจอง")
    booking_id_to_delete = st.number_input("กรอก 'รหัสการจอง' ที่ต้องการลบ", min_value=1, step=1)

    if st.button("ลบข้อมูล", type="primary"):
        booking_to_delete = next((b for b in st.session_state.bookings if b["booking_id"] == booking_id_to_delete), None)

        if booking_to_delete:
            st.session_state.bookings.remove(booking_to_delete)
            st.success(f"✅ ลบข้อมูลการจองรหัส {booking_id_to_delete} เรียบร้อยแล้ว")
        else:
            st.error(f"❌ ไม่พบข้อมูลการจองรหัส {booking_id_to_delete}")

Overwriting app.py


install ngrok

In [9]:
pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 101.5 MB/s eta 0:00:00


install pandas

In [20]:
pip install pandas

login ngrok

In [10]:
!ngrok authtoken 36Gu14zX9yFqEHJIEgQ7ZxVVzTl_3KGsRN5r2iEVjjm2tPhrJ

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [34]:
from pyngrok import ngrok

ngrok.kill()  # ปิด tunnel เก่า

public_url = ngrok.connect(8501)
print("🌍 Open your app here:", public_url)

🌍 Open your app here: NgrokTunnel: "https://perispherical-spissus-joellen.ngrok-free.dev" -> "http://localhost:8501"


รันสตรีม

In [35]:
!streamlit run app.py --server.port 8501 &


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.75.97.54:8501

2026-03-25 17:19:55.742 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-25 17:20:18.195 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


  Stopping...
